# DermReady environment setup

Select a GPU in **Runtime > Change runtime type** before running. This notebook installs dependencies, checks CPU/GPU execution, and prepares persistent output paths. It does not load SCIN or train a model. Google Drive authorization requires your account.


In [ ]:
from pathlib import Path
import subprocess, sys, os
repo = Path('/content/DermReady_')
if not (repo / '.git').exists():
    subprocess.run(['git', 'clone', 'https://github.com/taqi-ul-hassan/DermReady_.git', str(repo)], check=True)
os.chdir(repo)
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)


## Preserve the CUDA runtime

Install pinned non-PyTorch dependencies. The local CPU lock file must not be installed in Colab. Restart the runtime if requested after installation.


In [ ]:
import torch, torchvision
print('Existing CUDA pair:', torch.__version__, torchvision.__version__)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-colab.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'check'], check=True)


In [ ]:
subprocess.run([sys.executable, 'scripts/check_environment.py', '--require-gpu', '--output', '/content/environment_colab.json'], check=True)


## Persistent experiment outputs

Mount your own Drive. This cell writes only to a new DermReady folder. Training must use the generated configuration when implemented.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import yaml, json
persistent = Path('/content/drive/MyDrive/DermReady')
for folder in ['checkpoints', 'splits', 'reports', 'configs']:
    (persistent / folder).mkdir(parents=True, exist_ok=True)
probe = persistent / '.dermready_write_check'
probe.write_text('DermReady persistent storage check')
assert probe.read_text() == 'DermReady persistent storage check'
probe.unlink()
config = yaml.safe_load(Path('configs/experiment.yaml').read_text())
config['training']['checkpoint_dir'] = str(persistent / 'checkpoints')
config['data']['split_dir'] = str(persistent / 'splits')
config['reports_dir'] = str(persistent / 'reports')
config_path = persistent / 'configs/colab_runtime.yaml'
config_path.write_text(yaml.safe_dump(config, sort_keys=False))
(persistent / 'reports/environment_colab.json').write_text(Path('/content/environment_colab.json').read_text())
freeze = subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True)
(persistent / 'reports/requirements_colab_runtime.txt').write_text(freeze)
print('Persistence checked. Runtime config:', config_path)


## Completion

The environment is validated only when all cells pass. Save the notebook in your Drive. Use the persistent paths for future checkpoints/results; temporary Colab files may disappear. Pretrained-weight access and real-model inference remain later checks.
